In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("train")

input_w01  = pd.read_csv(DATA_DIR / "input_2023_w01.csv")
output_w01 = pd.read_csv(DATA_DIR / "output_2023_w01.csv")

print(input_w01.shape, output_w01.shape)
print(input_w01.head())
print(output_w01.head())


(285714, 23) (32088, 6)
      game_id  play_id  player_to_predict  nfl_id  frame_id play_direction  \
0  2023090700      101              False   54527         1          right   
1  2023090700      101              False   54527         2          right   
2  2023090700      101              False   54527         3          right   
3  2023090700      101              False   54527         4          right   
4  2023090700      101              False   54527         5          right   

   absolute_yardline_number player_name player_height  player_weight  ...  \
0                        42  Bryan Cook           6-1            210  ...   
1                        42  Bryan Cook           6-1            210  ...   
2                        42  Bryan Cook           6-1            210  ...   
3                        42  Bryan Cook           6-1            210  ...   
4                        42  Bryan Cook           6-1            210  ...   

          player_role      x      y     s   

In [2]:
# last pre-pass frame per player
throw_state = (
    input_w01
    .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
    .tail(1)          # last frame before throw
)

throw_state = throw_state.rename(columns={"frame_id": "frame_id_input"})


In [3]:
throw_state["num_frames_output"].describe()
throw_state["player_role"].value_counts()
throw_state["player_to_predict"].value_counts()


player_to_predict
False    7410
True     2679
Name: count, dtype: int64

In [4]:
train_w01 = output_w01.merge(
    throw_state,
    on=["game_id", "play_id", "nfl_id"],
    how="inner",
    suffixes=("_future", "_throw")
)

print(train_w01.shape)
train_w01.head()


(32088, 26)


,game_id,play_id,nfl_id,frame_id,x_future,y_future,player_to_predict,frame_id_input,play_direction,absolute_yardline_number,...,player_role,x_throw,y_throw,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,46137,1,56.22,17.28,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
1,2023090700,101,46137,2,56.63,16.88,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
2,2023090700,101,46137,3,57.06,16.46,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
3,2023090700,101,46137,4,57.48,16.02,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22
4,2023090700,101,46137,5,57.91,15.56,True,26,right,42,...,Defensive Coverage,55.82,17.67,5.34,1.8,134.17,184.99,21,63.259998,-0.22


In [5]:
# inspect future frame index vs number of frames to predict
train_w01[["frame_id", "num_frames_output"]].describe()

# this one is already correct:
train_w01[["x_future", "y_future"]].describe()


,x_future,y_future
count,32088.000000,32088.000000
mean,59.020316,26.530208
std,25.550084,13.249616
min,0.420000,0.590000
25%,40.567500,15.050000
50%,58.430000,26.190000
75%,77.000000,38.080000
max,119.380000,52.910000


In [6]:
# Every future frame_id should be within the num_frames_output
(train_w01["frame_id"] <= train_w01["num_frames_output"]).value_counts()

# How many future frames per player_to_predict?
train_w01.groupby("player_to_predict")["frame_id"].max().describe()


count     1.0
mean     94.0
std       NaN
min      94.0
25%      94.0
50%      94.0
75%      94.0
max      94.0
Name: frame_id, dtype: float64

In [7]:
import numpy as np

def normalize_direction(df):
    df = df.copy()
    mask = df["play_direction"] == "left"

    # Flip x and y for throw & future + ball landing
    df.loc[mask, "x_throw"]   = 120 - df.loc[mask, "x_throw"]
    df.loc[mask, "y_throw"]   = 53.3 - df.loc[mask, "y_throw"]
    df.loc[mask, "x_future"]  = 120 - df.loc[mask, "x_future"]
    df.loc[mask, "y_future"]  = 53.3 - df.loc[mask, "y_future"]
    df.loc[mask, "ball_land_x"] = 120 - df.loc[mask, "ball_land_x"]
    df.loc[mask, "ball_land_y"] = 53.3 - df.loc[mask, "ball_land_y"]

    # Also flip orientation / direction by 180°
    df.loc[mask, "dir"] = (df.loc[mask, "dir"] + 180) % 360
    df.loc[mask, "o"]   = (df.loc[mask, "o"] + 180) % 360

    df["play_direction"] = "right"
    return df

train_w01n = normalize_direction(train_w01)


In [8]:
import numpy as np

train = train_w01n.copy()

# Time since throw (seconds) – frame_id in output is 1..num_frames_output, 10 fps
train["t"] = train["frame_id"] / 10.0

# Velocity components at throw (dir is in degrees)
train["dir_rad"] = np.deg2rad(train["dir"])
train["vx0"] = train["s"] * np.cos(train["dir_rad"])
train["vy0"] = train["s"] * np.sin(train["dir_rad"])

# Constant-velocity predicted positions
train["x_pred_cv"] = train["x_throw"] + train["vx0"] * train["t"]
train["y_pred_cv"] = train["y_throw"] + train["vy0"] * train["t"]

# Compute RMSE only on players that are scored
mask = train["player_to_predict"]

rmse_cv = np.sqrt(
    np.mean(
        (train.loc[mask, "x_future"] - train.loc[mask, "x_pred_cv"])**2 +
        (train.loc[mask, "y_future"] - train.loc[mask, "y_pred_cv"])**2
    )
)
print("Constant-velocity RMSE (week 1):", rmse_cv)


Constant-velocity RMSE (week 1): 9.285200502823058


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("train")

def get_throw_state(input_df: pd.DataFrame) -> pd.DataFrame:
    throw_state = (
        input_df
        .sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
        .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
        .tail(1)
        .rename(columns={"frame_id": "frame_id_input"})
    )
    return throw_state

def normalize_direction(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    mask = df["play_direction"] == "left"

    df.loc[mask, "x_throw"]   = 120 - df.loc[mask, "x_throw"]
    df.loc[mask, "y_throw"]   = 53.3 - df.loc[mask, "y_throw"]
    df.loc[mask, "x_future"]  = 120 - df.loc[mask, "x_future"]
    df.loc[mask, "y_future"]  = 53.3 - df.loc[mask, "y_future"]
    df.loc[mask, "ball_land_x"] = 120 - df.loc[mask, "ball_land_x"]
    df.loc[mask, "ball_land_y"] = 53.3 - df.loc[mask, "ball_land_y"]

    df.loc[mask, "dir"] = (df.loc[mask, "dir"] + 180) % 360
    df.loc[mask, "o"]   = (df.loc[mask, "o"] + 180) % 360

    df["play_direction"] = "right"
    return df

def build_week_dataset(week: int) -> pd.DataFrame:
    input_df  = pd.read_csv(DATA_DIR / f"input_2023_w{week:02d}.csv")
    output_df = pd.read_csv(DATA_DIR / f"output_2023_w{week:02d}.csv")

    throw_state = get_throw_state(input_df)

    week_df = output_df.merge(
        throw_state,
        on=["game_id", "play_id", "nfl_id"],
        how="inner",
        suffixes=("_future", "_throw"),
    )

    # rename for clarity
    week_df = week_df.rename(columns={"x_future": "x_future",
                                      "y_future": "y_future"})
    week_df = normalize_direction(week_df)

    return week_df


In [10]:
all_weeks = [build_week_dataset(w) for w in range(1, 19)]
train_all = pd.concat(all_weeks, ignore_index=True)
print(train_all.shape)


(562936, 26)


In [11]:
train_all = train_all.copy()
train_all["t"] = train_all["frame_id"] / 10.0

train_all["dir_rad"] = np.deg2rad(train_all["dir"])
train_all["vx0"] = train_all["s"] * np.cos(train_all["dir_rad"])
train_all["vy0"] = train_all["s"] * np.sin(train_all["dir_rad"])

train_all["x_pred_cv"] = train_all["x_throw"] + train_all["vx0"] * train_all["t"]
train_all["y_pred_cv"] = train_all["y_throw"] + train_all["vy0"] * train_all["t"]

mask = train_all["player_to_predict"]
rmse_cv_all = np.sqrt(
    np.mean(
        (train_all.loc[mask, "x_future"] - train_all.loc[mask, "x_pred_cv"])**2 +
        (train_all.loc[mask, "y_future"] - train_all.loc[mask, "y_pred_cv"])**2
    )
)
print("Constant-velocity RMSE (all weeks):", rmse_cv_all)


Constant-velocity RMSE (all weeks): 8.163254398161516


In [12]:
train_all["dx_future"] = train_all["x_future"] - train_all["x_throw"]
train_all["dy_future"] = train_all["y_future"] - train_all["y_throw"]

# Ball-relative features
train_all["dist_ball"] = np.sqrt(
    (train_all["ball_land_x"] - train_all["x_throw"])**2 +
    (train_all["ball_land_y"] - train_all["y_throw"])**2
)
train_all["angle_to_ball"] = np.arctan2(
    train_all["ball_land_y"] - train_all["y_throw"],
    train_all["ball_land_x"] - train_all["x_throw"],
)


In [13]:
feature_cols = [
    "t",
    "x_throw", "y_throw",
    "s", "a",
    "vx0", "vy0",
    "dist_ball", "angle_to_ball",
    "num_frames_output",
]


In [14]:
train_ml = train_all[train_all["player_to_predict"]].reset_index(drop=True)


In [15]:
train_ml[feature_cols + ["dx_future", "dy_future"]].describe()


,t,x_throw,y_throw,s,a,vx0,vy0,dist_ball,angle_to_ball,num_frames_output,dx_future,dy_future
count,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000,562936.000000
mean,0.786540,67.020923,26.491414,4.811613,2.732318,-0.055878,2.876556,11.778709,-0.006234,14.730804,2.486389,-0.000334
std,0.592385,22.991527,11.787700,2.052455,1.465647,3.508296,2.603534,7.443833,1.543187,7.160791,3.991773,3.728389
min,0.100000,7.440000,0.690000,0.000000,0.000000,-10.049202,-6.342470,0.022361,-3.141593,5.000000,-13.130000,-22.780000
25%,0.400000,48.510000,16.250000,3.300000,1.580000,-2.662554,1.013957,6.263547,-1.300621,10.000000,0.240000,-1.760000
50%,0.700000,64.520000,26.300000,4.770000,2.550000,-0.031303,2.753296,9.762234,-0.000677,13.000000,1.110000,-0.010000
75%,1.100000,84.440000,36.800000,6.350000,3.730000,2.543443,4.658872,16.037687,1.262041,19.000000,3.210000,1.740000
max,9.400000,119.390000,52.620000,10.340000,8.360000,9.589657,10.309109,49.740375,3.141593,94.000000,31.050000,23.380000


In [16]:
cols_to_check = feature_cols + ["dx_future", "dy_future"]
print(train_ml[cols_to_check].isnull().sum())


t                    0
x_throw              0
y_throw              0
s                    0
a                    0
vx0                  0
vy0                  0
dist_ball            0
angle_to_ball        0
num_frames_output    0
dx_future            0
dy_future            0
dtype: int64


In [17]:
from sklearn.model_selection import train_test_split
import numpy as np

# Unique games
game_ids = train_ml["game_id"].unique()

train_games, val_games = train_test_split(
    game_ids,
    test_size=0.2,
    random_state=42,
)

train_mask = train_ml["game_id"].isin(train_games)
val_mask   = train_ml["game_id"].isin(val_games)

X_train = train_ml.loc[train_mask, feature_cols]
X_val   = train_ml.loc[val_mask,   feature_cols]

y_train_dx = train_ml.loc[train_mask, "dx_future"]
y_train_dy = train_ml.loc[train_mask, "dy_future"]
y_val_dx   = train_ml.loc[val_mask,   "dx_future"]
y_val_dy   = train_ml.loc[val_mask,   "dy_future"]

len(X_train), len(X_val)


(445478, 117458)

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf_dx = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)
rf_dy = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)

rf_dx.fit(X_train, y_train_dx)
rf_dy.fit(X_train, y_train_dy)

dx_pred_val = rf_dx.predict(X_val)
dy_pred_val = rf_dy.predict(X_val)

# RMSE in displacement space (same as in x,y space)
rmse_model = np.sqrt(
    np.mean((y_val_dx - dx_pred_val)**2 + (y_val_dy - dy_pred_val)**2)
)
print("RandomForest RMSE (val):", rmse_model)


RandomForest RMSE (val): 1.7667896329596224


In [19]:
# Indices of validation rows (in train_ml / train_all)
val_rows = train_ml.loc[val_mask].index

x_true = train_all.loc[val_rows, "x_future"]
y_true = train_all.loc[val_rows, "y_future"]

x_cv   = train_all.loc[val_rows, "x_pred_cv"]
y_cv   = train_all.loc[val_rows, "y_pred_cv"]

rmse_cv_val = np.sqrt(np.mean((x_true - x_cv)**2 + (y_true - y_cv)**2))
print("Constant-velocity RMSE (val):", rmse_cv_val)
print("RandomForest RMSE (val):", rmse_model)
print("Improvement factor:", rmse_cv_val / rmse_model)


Constant-velocity RMSE (val): 8.712156687660606
RandomForest RMSE (val): 1.7667896329596224
Improvement factor: 4.931066226071585


In [20]:
import pandas as pd

fi_dx = pd.Series(rf_dx.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi_dy = pd.Series(rf_dy.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("Feature importance for dx:")
print(fi_dx)

print("\nFeature importance for dy:")
print(fi_dy)


Feature importance for dx:
t                    0.558983
vy0                  0.345312
angle_to_ball        0.037144
dist_ball            0.018623
num_frames_output    0.010411
a                    0.009650
y_throw              0.007189
vx0                  0.004484
s                    0.004346
x_throw              0.003858
dtype: float64

Feature importance for dy:
vx0                  0.605963
t                    0.231469
angle_to_ball        0.107688
y_throw              0.015286
a                    0.010209
dist_ball            0.009743
s                    0.006446
vy0                  0.005142
x_throw              0.004161
num_frames_output    0.003894
dtype: float64


In [32]:
import xgboost as xgb
import numpy as np

In [35]:
import xgboost as xgb
import numpy as np

params_common = dict(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
)

xgb_dx = xgb.XGBRegressor(**params_common)
xgb_dy = xgb.XGBRegressor(**params_common)

# dx model
xgb_dx.fit(
    X_train, y_train_dx,
    eval_set=[(X_val, y_val_dx)],
    verbose=False,      # you can set True to see eval metrics each round
)

# dy model
xgb_dy.fit(
    X_train, y_train_dy,
    eval_set=[(X_val, y_val_dy)],
    verbose=False,
)

dx_pred_val_xgb = xgb_dx.predict(X_val)
dy_pred_val_xgb = xgb_dy.predict(X_val)

rmse_xgb = np.sqrt(
    np.mean(
        (y_val_dx - dx_pred_val_xgb) ** 2 +
        (y_val_dy - dy_pred_val_xgb) ** 2
    )
)
print("XGBoost RMSE (val):", rmse_xgb)


XGBoost RMSE (val): 1.642850801644632


In [36]:
print("Constant-velocity RMSE (val):", rmse_cv_val)
print("RandomForest RMSE (val):     ", rmse_model)
print("XGBoost RMSE (val):          ", rmse_xgb)

print("\nRF vs CV improvement:", rmse_cv_val / rmse_model)
print("XGB vs CV improvement:", rmse_cv_val / rmse_xgb)
print("XGB vs RF improvement:", rmse_model / rmse_xgb)


Constant-velocity RMSE (val): 8.712156687660606
RandomForest RMSE (val):      1.7667896329596224
XGBoost RMSE (val):           1.642850801644632

RF vs CV improvement: 4.931066226071585
XGB vs CV improvement: 5.30307236599879
XGB vs RF improvement: 1.0754413189505203


In [37]:
import pandas as pd

fi_dx_xgb = pd.Series(xgb_dx.feature_importances_, index=feature_cols).sort_values(ascending=False)
fi_dy_xgb = pd.Series(xgb_dy.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("XGBoost feature importance for dx:")
print(fi_dx_xgb)

print("\nXGBoost feature importance for dy:")
print(fi_dy_xgb)


XGBoost feature importance for dx:
vy0                  0.396615
t                    0.329946
num_frames_output    0.111331
s                    0.056453
angle_to_ball        0.040116
vx0                  0.022964
dist_ball            0.016985
a                    0.013614
y_throw              0.007661
x_throw              0.004313
dtype: float32

XGBoost feature importance for dy:
vx0                  0.616055
angle_to_ball        0.131780
t                    0.130726
num_frames_output    0.032025
y_throw              0.031546
s                    0.016941
vy0                  0.013481
dist_ball            0.012064
a                    0.011406
x_throw              0.003976
dtype: float32


In [38]:
import numpy as np
import pandas as pd

def parse_height(h):
    if isinstance(h, str) and "-" in h:
        ft, inch = h.split("-")
        try:
            return int(ft) * 12 + int(inch)
        except ValueError:
            return np.nan
    return np.nan

# Height in inches
train_ml["height_in"] = train_ml["player_height"].apply(parse_height)

# Weight already numeric-ish, but ensure float
train_ml["weight_lb"] = pd.to_numeric(train_ml["player_weight"], errors="coerce")

# Rough age in years (2023 season vs birth year)
birth_year = pd.to_datetime(train_ml["player_birth_date"], errors="coerce").dt.year
train_ml["age_years"] = 2023 - birth_year

# Context feature
train_ml["abs_yardline"] = train_ml["absolute_yardline_number"].astype(float)


In [39]:
train_ml[["height_in", "weight_lb", "age_years", "abs_yardline"]].describe()


,height_in,weight_lb,age_years,abs_yardline
count,562936.000000,562936.000000,562936.000000,562936.000000
mean,72.790729,208.435440,26.144794,60.359034
std,2.084684,21.713407,2.845624,23.100056
min,66.000000,153.000000,21.000000,11.000000
25%,71.000000,193.000000,24.000000,41.000000
50%,73.000000,203.000000,26.000000,60.000000
75%,74.000000,220.000000,28.000000,79.000000
max,81.000000,358.000000,39.000000,109.000000


In [40]:
cat_cols = ["player_role", "player_position", "player_side"]

train_ml_enc = pd.get_dummies(train_ml, columns=cat_cols, drop_first=True)

# collect new dummy column names
dummy_cols = [
    c for c in train_ml_enc.columns
    if c.startswith("player_role_")
    or c.startswith("player_position_")
    or c.startswith("player_side_")
]


In [41]:
base_numeric = feature_cols + ["height_in", "weight_lb", "age_years", "abs_yardline"]

feature_cols_ext = base_numeric + dummy_cols

# re-use the same train_mask / val_mask you already computed
X_train_ext = train_ml_enc.loc[train_mask, feature_cols_ext]
X_val_ext   = train_ml_enc.loc[val_mask,   feature_cols_ext]

y_train_dx = train_ml_enc.loc[train_mask, "dx_future"]
y_train_dy = train_ml_enc.loc[train_mask, "dy_future"]
y_val_dx   = train_ml_enc.loc[val_mask,   "dx_future"]
y_val_dy   = train_ml_enc.loc[val_mask,   "dy_future"]


In [42]:
import xgboost as xgb
import numpy as np

params_common = dict(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
)

xgb_dx_ext = xgb.XGBRegressor(**params_common)
xgb_dy_ext = xgb.XGBRegressor(**params_common)

xgb_dx_ext.fit(X_train_ext, y_train_dx, eval_set=[(X_val_ext, y_val_dx)], verbose=False)
xgb_dy_ext.fit(X_train_ext, y_train_dy, eval_set=[(X_val_ext, y_val_dy)], verbose=False)

dx_pred_val_xgb_ext = xgb_dx_ext.predict(X_val_ext)
dy_pred_val_xgb_ext = xgb_dy_ext.predict(X_val_ext)

rmse_xgb_ext = np.sqrt(
    np.mean(
        (y_val_dx - dx_pred_val_xgb_ext)**2 +
        (y_val_dy - dy_pred_val_xgb_ext)**2
    )
)

print("Old XGBoost RMSE (val):", rmse_xgb)
print("New XGBoost RMSE (val) with extra features:", rmse_xgb_ext)
print("Improvement factor vs old XGB:", rmse_xgb / rmse_xgb_ext)


Old XGBoost RMSE (val): 1.642850801644632
New XGBoost RMSE (val) with extra features: 1.5964190128084126
Improvement factor vs old XGB: 1.0290849635739032


In [43]:
def normalize_direction_inference(df: pd.DataFrame):
    """Flip plays so offense goes left->right, but remember which were flipped."""
    df = df.copy()
    mask_left = df["play_direction"] == "left"

    # Flip throw + ball landing
    df.loc[mask_left, "x_throw"]     = 120 - df.loc[mask_left, "x_throw"]
    df.loc[mask_left, "y_throw"]     = 53.3 - df.loc[mask_left, "y_throw"]
    df.loc[mask_left, "ball_land_x"] = 120 - df.loc[mask_left, "ball_land_x"]
    df.loc[mask_left, "ball_land_y"] = 53.3 - df.loc[mask_left, "ball_land_y"]

    df.loc[mask_left, "dir"] = (df.loc[mask_left, "dir"] + 180) % 360
    df.loc[mask_left, "o"]   = (df.loc[mask_left, "o"] + 180) % 360

    df.loc[mask_left, "play_direction"] = "right"
    return df, mask_left


In [44]:
def add_player_numeric_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["height_in"] = df["player_height"].apply(parse_height)
    df["weight_lb"] = pd.to_numeric(df["player_weight"], errors="coerce")

    birth_year = pd.to_datetime(df["player_birth_date"], errors="coerce").dt.year
    df["age_years"] = 2023 - birth_year

    df["abs_yardline"] = df["absolute_yardline_number"].astype(float)

    return df


In [48]:
def build_test_features(test_input: pd.DataFrame,
                        test_df: pd.DataFrame):
    """
    Build feature matrix for test rows (same features as training).

    Returns:
      merged_enc: full encoded dataframe (with x_throw etc.)
      X_test:     feature matrix matching feature_cols_ext
      mask_left:  boolean mask of rows that were originally left-going plays
    """
    # 1) last pre-pass frame per player in this test chunk
    throw_state = get_throw_state(test_input)

    # 2) join test targets (game_id, play_id, nfl_id, frame_id) with throw_state
    merged = test_df.merge(
        throw_state,
        on=["game_id", "play_id", "nfl_id"],
        how="left",
    )

    # keep original direction for later unflip (optional, but nice to have)
    merged["orig_play_direction"] = merged["play_direction"]

    # 3) rename x,y from throw_state to x_throw,y_throw (match training)
    merged = merged.rename(columns={
        "x": "x_throw",
        "y": "y_throw",
    })

    # 4) add numeric player features (height, weight, age, yardline)
    merged = add_player_numeric_features(merged)

    # 5) normalize direction and remember which rows were flipped
    merged, mask_left = normalize_direction_inference(merged)

    # 6) kinematic + ball features (same as training)
    merged["t"] = merged["frame_id"] / 10.0
    merged["dir_rad"] = np.deg2rad(merged["dir"])
    merged["vx0"] = merged["s"] * np.cos(merged["dir_rad"])
    merged["vy0"] = merged["s"] * np.sin(merged["dir_rad"])

    merged["dist_ball"] = np.sqrt(
        (merged["ball_land_x"] - merged["x_throw"])**2 +
        (merged["ball_land_y"] - merged["y_throw"])**2
    )
    merged["angle_to_ball"] = np.arctan2(
        merged["ball_land_y"] - merged["y_throw"],
        merged["ball_land_x"] - merged["x_throw"],
    )

    # 7) one-hot encode categoricals and align dummy columns
    merged_enc = pd.get_dummies(merged, columns=cat_cols, drop_first=True)

    # make sure all training dummy columns exist
    for col in dummy_cols:
        if col not in merged_enc.columns:
            merged_enc[col] = 0

    # ensure feature order matches training
    X_test = merged_enc[feature_cols_ext].astype(float)

    return merged_enc, X_test, mask_left


In [49]:
def predict_on_test_like_kaggle(test_input_path: str, test_path: str) -> pd.DataFrame:
    test_input = pd.read_csv(test_input_path)
    test_df    = pd.read_csv(test_path)

    merged_enc, X_test, mask_left = build_test_features(test_input, test_df)

    # Predict dx, dy in canonical coordinates
    dx_pred = xgb_dx_ext.predict(X_test)
    dy_pred = xgb_dy_ext.predict(X_test)

    # Reconstruct x,y in canonical orientation
    x_canon = merged_enc["x_throw"] + dx_pred
    y_canon = merged_enc["y_throw"] + dy_pred

    # Unflip plays that were originally going left
    x_pred = x_canon.copy()
    y_pred = y_canon.copy()

    x_pred[mask_left] = 120 - x_canon[mask_left]
    y_pred[mask_left] = 53.3 - y_canon[mask_left]

    # Build submission-like dataframe
    preds = test_df.copy()
    preds["x"] = x_pred.values
    preds["y"] = y_pred.values

    return preds[["id", "x", "y"]]


In [50]:
preds_mock = predict_on_test_like_kaggle("test_input.csv", "test.csv")
preds_mock.head()


,id,x,y
0,12350,88.296547,34.388861
1,12351,88.494647,34.352287
2,12352,88.784550,34.416054
3,12353,89.005947,34.452273
4,12354,89.320017,34.542149
